In [1]:
# =============================================================================
# CHUNK 1 — IMPORTS + GLOBAL CONFIG + LOAD DATASET
# =============================================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords

sns.set(style="whitegrid")

# Global seed
Randomizer = 485
np.random.seed(Randomizer)

# Load cleaned dataset
WineReview_A = pd.read_csv("data/WineReview_A_cleaned.csv")

# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# =============================================================================
# CHUNK 2 — FIX SENTIMENT COLUMN
# =============================================================================

WineReview_A["sentiment"] = (
    WineReview_A["sentiment"]
    .astype(str)
    .str.strip()
    .replace({
        "1": "positive",
        "-1": "negative",
        "0": "neutral"
    })
)

In [3]:
# =============================================================================
# CHUNK 3 — TOKENIZER (MATCHES R'S unnest_tokens)
# =============================================================================

def tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    tokens = text.split()
    return [t for t in tokens if t not in stop_words]

In [4]:
# =============================================================================
# CHUNK 4 — TOKENIZE FULL DATASET
# =============================================================================

words_df = (
    WineReview_A
    .assign(word=lambda df: df["clean_text"].apply(tokenize))
    .explode("word")
)

In [5]:
# =============================================================================
# CHUNK 5 — WORD FREQUENCY + AVG RATING
# =============================================================================

word_rating = (
    words_df.groupby("word")
    .agg(
        avg_rating=("rating_points", "mean"),
        freq=("word", "count")
    )
    .query("freq >= 20")
    .sort_values("avg_rating", ascending=False)
    .reset_index()
)

In [6]:
# =============================================================================
# CHUNK 6 — TOP 20 POSITIVE WORDS
# =============================================================================

positive_words = (
    WineReview_A[WineReview_A["sentiment"] == "positive"]
    .assign(word=lambda df: df["clean_text"].apply(tokenize))
    .explode("word")
    .groupby("word")
    .size()
    .reset_index(name="freq_pos")
)

positive_word_rating = (
    positive_words.merge(word_rating, on="word", how="inner")
    .assign(
        freq=lambda df: df["freq_pos"],
        sentiment_group="positive"
    )[["word", "freq", "avg_rating", "sentiment_group"]]
    .sort_values("avg_rating", ascending=False)
    .head(20)
)

In [7]:
# =============================================================================
# CHUNK 7 — TOP 20 NEGATIVE WORDS
# =============================================================================

negative_words = (
    WineReview_A[WineReview_A["sentiment"] == "negative"]
    .assign(word=lambda df: df["clean_text"].apply(tokenize))
    .explode("word")
    .groupby("word")
    .size()
    .reset_index(name="freq_neg")
)

negative_word_rating = (
    negative_words.merge(word_rating, on="word", how="inner")
    .assign(
        freq=lambda df: df["freq_neg"],
        sentiment_group="negative"
    )[["word", "freq", "avg_rating", "sentiment_group"]]
    .sort_values("avg_rating", ascending=True)
    .head(20)
)

In [ ]:
# =============================================================================
# CHUNK 10 — PRINT TOP WORD TABLES
# =============================================================================

print("Top 20 Positive Words")
display(positive_word_rating)

print("\nTop 20 Negative Words")
display(negative_word_rating)

Top 20 Positive Words


,word,freq,avg_rating,sentiment_group
764,cayuse,19,95.050000,positive
1565,endless,47,94.122449,positive
2432,incisive,25,94.040000,positive
4727,stunningly,34,94.000000,positive
5128,ultrafine,19,93.750000,positive
1734,extraordinary,69,93.750000,positive
5446,wondrous,33,93.666667,positive
2183,greatest,56,93.661017,positive
2708,leonetti,20,93.450000,positive
4726,stunning,260,93.413919,positive



Top 20 Negative Words


,word,freq,avg_rating,sentiment_group
4476,weird,45,82.509804,negative
19,acceptable,57,82.647887,negative
4276,underdeveloped,22,82.884615,negative
3831,stale,23,82.892857,negative
3464,sauerkraut,16,82.909091,negative
133,ammonia,17,83.000000,negative
3665,sketchy,33,83.195122,negative
375,bland,174,83.281106,negative
4455,watery,144,83.335000,negative
3894,strange,96,83.373134,negative


: 